In [4]:
display(spark.sql("SHOW TABLES"))

import pandas as pd
from datetime import datetime

df = spark.sql("SELECT * FROM b_sales_transactions").toPandas()

issues = []
print("=" * 50)
print("DATA QUALITY REPORT - Sales Transactions")
print(f"Run Time: {datetime.now()}")
print("=" * 50)


total_rows = len(df)
print(f"\n✅ Total Records: {total_rows}")
if total_rows != 200:
    issues.append(f"⚠️ Expected 200 rows, found {total_rows}")



print("\n── Null Value Check ──")
nulls = df.isnull().sum()
for col, count in nulls.items():
    if count > 0:
        print(f"  ⚠️ {col}: {count} nulls")
        issues.append(f"Null values in {col}: {count}")
    else:
        print(f"  ✅ {col}: No nulls")



print("\n── Duplicate Check ──")
dupes = df['transaction_id'].duplicated().sum()
if dupes > 0:
    print(f"  ⚠️ {dupes} duplicate transaction_ids found!")
    issues.append(f"Duplicate transaction_ids: {dupes}")
else:
    print(f"  ✅ No duplicate transaction_ids")



print("\n── Negative Values Check ──")
for col in ['unit_price', 'quantity', 'discount_pct', 'shipping_cost']:
    neg = (df[col] < 0).sum()
    if neg > 0:
        print(f"  ⚠️ {col}: {neg} negative values!")
        issues.append(f"Negative values in {col}: {neg}")
    else:
        print(f"  ✅ {col}: No negative values")



print("\n── Valid Values Check ──")

valid_status = ['Delivered', 'Shipped', 'Processing', 'Returned']
invalid_status = df[~df['order_status'].isin(valid_status)]
if len(invalid_status) > 0:
    print(f"  ⚠️ Invalid order_status values: {invalid_status['order_status'].unique()}")
    issues.append(f"Invalid order_status found")
else:
    print(f"  ✅ order_status: All valid")

valid_segments = ['Corporate', 'Consumer', 'Small Business']
invalid_seg = df[~df['customer_segment'].isin(valid_segments)]
if len(invalid_seg) > 0:
    print(f"  ⚠️ Invalid customer_segment: {invalid_seg['customer_segment'].unique()}")
    issues.append("Invalid customer_segment found")
else:
    print(f"  ✅ customer_segment: All valid")



print("\n── Date Format Check ──")
try:
    pd.to_datetime(df['order_date'], format='%d-%m-%Y')
    print("  ✅ order_date: All valid dates")
except:
    print("  ⚠️ order_date: Some invalid date formats!")
    issues.append("Invalid date formats in order_date")



print("\n" + "=" * 50)
if len(issues) == 0:
    print("🎉 ALL CHECKS PASSED! Data is clean.")
else:
    print(f"⚠️ {len(issues)} issue(s) found:")
    for i in issues:
        print(f"  - {i}")
print("=" * 50)




from pyspark.sql import Row
from datetime import datetime

audit_data = Row(
    run_timestamp = str(datetime.now()),
    total_rows_checked = int(total_rows),
    null_issues = int(nulls.sum()),
    duplicate_issues = int(dupes),
    total_issues_found = int(len(issues)),
    status = "PASSED" if len(issues) == 0 else "FAILED"
)

audit_df = spark.createDataFrame([audit_data])

audit_df.write.mode("append").saveAsTable("audit_log")

print("✅ Audit log saved successfully!")

StatementMeta(, 6ec8404a-abe8-48c9-8f17-448a1cf26112, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2ed26b59-b3f8-4f51-afbc-052f385b65e2)

DATA QUALITY REPORT - Sales Transactions
Run Time: 2026-05-30 10:49:56.183202

✅ Total Records: 200

── Null Value Check ──
  ✅ transaction_id: No nulls
  ✅ order_date: No nulls
  ✅ customer_id: No nulls
  ✅ customer_name: No nulls
  ✅ customer_email: No nulls
  ✅ customer_segment: No nulls
  ✅ region: No nulls
  ✅ country: No nulls
  ✅ city: No nulls
  ✅ product_id: No nulls
  ✅ product_name: No nulls
  ✅ category: No nulls
  ✅ sub_category: No nulls
  ✅ unit_price: No nulls
  ✅ quantity: No nulls
  ✅ discount_pct: No nulls
  ✅ shipping_cost: No nulls
  ✅ payment_method: No nulls
  ✅ order_status: No nulls
  ✅ sales_rep: No nulls

── Duplicate Check ──
  ✅ No duplicate transaction_ids

── Negative Values Check ──
  ✅ unit_price: No negative values
  ✅ quantity: No negative values
  ✅ discount_pct: No negative values
  ✅ shipping_cost: No negative values

── Valid Values Check ──
  ✅ order_status: All valid
  ✅ customer_segment: All valid

── Date Format Check ──
  ✅ order_date: All va